# UAV 路径规划 — 交互式操作面板

**使用规则：每次打开 Notebook 后，必须先运行第一格（初始化），之后所有格均可独立点击运行。**

---

In [2]:
# ★ 初始化（每次打开 Notebook 后必须首先运行此格）
from pipeline_api import *

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
✅ pipeline_api 加载完成。
   常用指令: run() | status() | summary() | show_config() | help(<函数名>)


---
## 一、运行控制

| 指令 | 说明 |
|---|---|
| `run()` | 完整运行全部 5 个阶段 |
| `run("3-5")` | 从阶段3开始运行到阶段5 |
| `run(3)` | 只运行阶段3 |
| `run("2", force=True)` | 强制重新计算阶段2（忽略已有检查点） |
| `stage1()` ~ `stage5()` | 单独运行某一阶段 |
| `stage3(force=True, quality_threshold=0.7)` | 带参数运行阶段3 |

In [ ]:
run()   # 完整运行（阶段1 → 5）

In [ ]:
run("3-5")   # 从阶段3恢复运行（需要阶段2的检查点）

In [2]:
stage1()   # 单独运行阶段1


──────────────────────────────────────────────────────
  阶段1 / 5 : 网格预处理 & 特征点提取
──────────────────────────────────────────────────────

[Module 1] 启动预处理与特征提取 (data/airplane_aligned.stl)...
 -> 正在清理底层网格并提取顶点法向...
 -> 正在执行向量化 PCA 曲率计算与自适应采样...
    KDTree 批量近邻查询完成，耗时 1.3s
[Module 1] 预处理完成！特征点数: 61171  耗时: 7.6s
  [✓] 检查点已保存: output/checkpoints/stage_1.pkl  (2.9 MB)
  耗时: 7.6s


{'pts': array([[ -5.35749128,   2.3624379 ,   4.35224581],
        [ -1.67718668, -10.42841662,   4.94171346],
        [  6.75702945,   2.04066568,   3.35339845],
        ...,
        [  0.8731061 , -17.83009931,   6.39495498],
        [  1.87819517,  11.99779832,   4.90526899],
        [ -7.03329255,   7.22836346,   3.53064433]], shape=(61171, 3)),
 'norms': array([[ 0.06425407, -0.05841559,  0.99622052],
        [-0.97097409, -0.12130466, -0.20603538],
        [-0.46723198, -0.30960268, -0.82254489],
        ...,
        [-0.23531281, -0.33044348,  0.90983351],
        [ 0.99176444,  0.0267498 ,  0.12509325],
        [ 0.57769397,  0.24901213, -0.77167377]], shape=(61171, 3))}

In [ ]:
stage2()   # 单独运行阶段2（耗时最长，建议完成后不要轻易重跑）


──────────────────────────────────────────────────────
  阶段2 / 5 : 候选视点生成 & 覆盖质量评估
──────────────────────────────────────────────────────
[Module 2] 正在提取底层 CAD 网格的绝对物理面元属性...

[Module 2] 启动距离与极坐标交替试探算法 (1对1优选池)...
 -> [距离探测] 尝试拉远 0.0 米...
 -> [距离探测] 尝试拉远 0.5 米...
 -> [距离探测] 尝试拉远 1.0 米...
 -> [距离探测] 尝试拉远 1.5 米...
 -> [距离探测] 尝试拉远 2.0 米...
 -> 提纯后候选视点池: 16221 个。
 -> 正在启动高并发视锥射线阵列，计算真实三角面元覆盖与物理质量矩阵...
✔️ 数字相机扫描完毕！耗时: 134.25 秒
✔️ 提纯后有效视点数: 16220
📊 飞机总表面积: 1000.72 m² | 实际覆盖面积: 999.26 m²
🌟 真实物理表面覆盖率: 99.85%
[Module 2] 总耗时: 136.59 秒


In [3]:
stage3()   # 单独运行阶段3（使用默认覆盖阈值 0.85）


──────────────────────────────────────────────────────
  阶段3 / 5 : 质量感知集合覆盖优化
──────────────────────────────────────────────────────
  [✓] 检查点已加载: output/checkpoints/stage_2.pkl  (1324.8 MB)

[Module 3] 正在解析质量矩阵与计算物理达标阈值...
 -> 矩阵解析完成！待优选候选视点: 16220 个 | 待覆盖三角面元: 354159 个

 🗜️ 启动模块 3: 极速张量化 Lazy Greedy 次模边缘贪心优化
  -> 正在初始化张量优先堆...

 🏁 质量感知航点精简完成 | 最终胜出航点数: 1314 个
 💡 压缩率: 选出了全集 8.10% 的精英视点
 ⏱️ 核心算法耗时: 12.01 秒
  [✓] 检查点已保存: output/checkpoints/stage_3.pkl  (0.0 MB)
  精选航点: 1314  耗时: 116.0s


{'final_waypoints': array([[ 5.47289855,  1.72483744,  9.08869012],
        [-5.55353674,  2.3710764 ,  8.94175259],
        [ 6.13757769,  9.77899118,  8.21290218],
        ...,
        [ 6.41254869, -9.79471881,  7.09876089],
        [-9.55899331,  5.73881871,  0.61137863],
        [-1.3781841 , -7.38042245, 11.87999692]], shape=(1314, 3))}

In [ ]:
stage3(force=True, quality_threshold=0.7)   # 强制重跑，调低阈值（航点更少）

In [4]:
stage4()   # 单独运行阶段4（多机任务分配 + TSP）


──────────────────────────────────────────────────────
  阶段4 / 5 : 多机任务分配 & TSP 拓扑排序
──────────────────────────────────────────────────────
[Module 4] 正在执行动力学感知聚类 (K=4)...

[阶段 1] 正在进行拓扑排序与直线可视化...
 -> 规划 UAV 1 的路径...
 -> 构建 4D 代价矩阵 (节点数: 289)...
 -> 规划 UAV 2 的路径...
 -> 构建 4D 代价矩阵 (节点数: 363)...
 -> 规划 UAV 3 的路径...
 -> 构建 4D 代价矩阵 (节点数: 369)...
 -> 规划 UAV 4 的路径...
 -> 构建 4D 代价矩阵 (节点数: 297)...
 已导出: output/visualizations/4_topo_lines.ply
  [✓] 检查点已保存: output/checkpoints/stage_4.pkl  (0.0 MB)
  4 架无人机路线已生成  耗时: 52.2s


{'all_routes': [(array([[ 2.00000000e+01,  2.00000000e+01,  5.00000000e-01],
          [ 9.17306972e+00,  8.71609992e+00,  6.34116233e-01],
          [ 7.51760341e+00,  7.39494097e+00,  5.65440331e-01],
          [ 7.08307020e+00,  8.33526006e+00,  8.47580656e-01],
          [ 5.95985436e+00,  7.85248243e+00,  8.11167809e-01],
          [ 6.52774472e+00,  8.49625252e+00,  5.51423888e-01],
          [ 5.59819179e+00,  9.55366753e+00,  5.99438855e-01],
          [ 4.14543739e+00,  1.01981493e+01,  8.35345989e-01],
          [ 3.53838335e+00,  8.70665538e+00,  9.79453888e-01],
          [ 3.56249665e+00,  7.87160457e+00,  7.98986431e-01],
          [ 4.19349741e+00,  6.74175290e+00,  1.42648152e+00],
          [ 4.36617289e+00,  6.66836149e+00,  1.67254987e+00],
          [ 4.54046438e+00,  7.73809578e+00,  2.53653724e+00],
          [ 4.53437498e+00,  7.80206532e+00,  3.05227548e+00],
          [ 5.07281629e+00,  8.13745061e+00,  2.87975648e+00],
          [ 6.14056610e+00,  8.40636917e+

In [5]:
stage5()   # 单独运行阶段5（A* 避障 + 轨迹 CSV 导出）


──────────────────────────────────────────────────────
  阶段5 / 5 : 避障轨迹优化 & CSV 导出
──────────────────────────────────────────────────────

[Module 5 - CollisionChecker] 正在构建碰撞检测场...
  -> 碰撞场构建完成。全程飞行安全半径: 2.0 m

[Module 5 - VoxelAStarPlanner] 正在构建体素安全图...
  -> 体素分辨率: 0.5 m | 网格尺寸: [92, 97, 42] = 374808 个体素
  -> 安全体素图构建完成！安全/总计 = 282052/374808 (75.3%) | 耗时: 1.9s

[UAV 1] 开始构建轨迹 (共 290 个航点，289 段)...
  [Seg   1/289] [20.  20.   0.5] -> [9.2 8.7 0.6] ...
  [Seg   2/289] [9.2 8.7 0.6] -> [7.5 7.4 0.6] ...
  [Seg   3/289] [7.5 7.4 0.6] -> [7.1 8.3 0.8] ...
  [Seg   4/289] [7.1 8.3 0.8] -> [6.  7.9 0.8] ...
  [Seg   5/289] [6.  7.9 0.8] -> [6.5 8.5 0.6] ...
  [Seg   6/289] [6.5 8.5 0.6] -> [5.6 9.6 0.6] ...
  [Seg   7/289] [5.6 9.6 0.6] -> [ 4.1 10.2  0.8] ...
  [Seg   8/289] [ 4.1 10.2  0.8] -> [3.5 8.7 1. ] ...
  [Seg   9/289] [3.5 8.7 1. ] -> [3.6 7.9 0.8] ...
  [Seg  10/289] [3.6 7.9 0.8] -> [4.2 6.7 1.4] ...
  [Seg  11/289] [4.2 6.7 1.4] -> [4.4 6.7 1.7] ...
  [Seg  12/289] [4.4 6.7 1.7

[[(0.0, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF_START'),
  (0.1, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF'),
  (0.2, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF'),
  (0.3, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF'),
  (0.4, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF'),
  (0.5, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF'),
  (0.6, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF'),
  (0.7, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF'),
  (0.8, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF'),
  (0.9, 20.0, 20.0, 0.5, 0.0, 'TAKEOFF_END'),
  (1.0, 20.0, 20.0, 0.5, 0.0, 'TRANSIT'),
  (1.1, 19.8594, 19.8535, 0.5017, 0.0, 'TRANSIT'),
  (1.2, 19.7188, 19.7069, 0.5035, 0.0, 'TRANSIT'),
  (1.3, 19.5782, 19.5604, 0.5052, 0.0, 'TRANSIT'),
  (1.4, 19.4376, 19.4138, 0.507, 0.0, 'TRANSIT'),
  (1.5, 19.297, 19.2673, 0.5087, 0.0, 'TRANSIT'),
  (1.6, 19.1563, 19.1207, 0.5105, 0.0, 'TRANSIT'),
  (1.7, 19.0157, 18.9742, 0.5122, 0.0, 'TRANSIT'),
  (1.8, 18.8751, 18.8276, 0.5139, 0.0, 'TRANSIT'),
  (1.9, 18.7345, 18.6811, 0.5157, 0.0, 'TRANSIT'),
  (2.0, 18.5939, 18.5346, 0.5174, 0.0, 'TRANSIT'),
  (2.1, 18.4533, 18.

---
## 二、状态与结果查看

| 指令 | 说明 |
|---|---|
| `status()` | 查看所有检查点文件和输出文件的存在状态 |
| `summary()` | 查看各阶段的关键结果数字（航点数、路径长度等）|
| `inspect_csv(1)` | 查看 UAV 1 的轨迹 CSV 统计信息 |
| `inspect_csv(2, n_rows=10)` | 查看 UAV 2 的 CSV，预览前10行 |

In [6]:
status()   # 查看所有检查点和输出文件状态


────────────────────────────────────────────────────────────
  检查点 & 输出状态
────────────────────────────────────────────────────────────
  ✅ 阶段1  网格预处理 & 特征点提取
      output/checkpoints/stage_1.pkl  2.9 MB  04-22 18:40
  ✅ 阶段2  候选视点生成 & 覆盖质量评估
      output/checkpoints/stage_2.pkl  1324.8 MB  04-22 18:49  [已缓存]
  ✅ 阶段3  质量感知集合覆盖优化
      output/checkpoints/stage_3.pkl  0.0 MB  04-22 19:08  [已缓存]
  ✅ 阶段4  多机任务分配 & TSP 拓扑排序
      output/checkpoints/stage_4.pkl  0.0 MB  04-22 19:12  [已缓存]
  ✅  output/trajectories/  (4 个 CSV)  阶段5  避障轨迹优化 & CSV 导出

  PLY 文件 (output/visualizations/): 3_final_waypoints.ply, 4_topo_lines.ply, 5_final_trajectories.ply
────────────────────────────────────────────────────────────



In [7]:
summary()   # 查看各阶段结果摘要（航点数、路径长度、CSV 统计等）


────────────────────────────────────────────────────
  流水线结果摘要
────────────────────────────────────────────────────
  [✓] 检查点已加载: output/checkpoints/stage_1.pkl  (2.9 MB)
  阶段1  特征点: 61,171  Z=[1.1, 13.0] m
  阶段2  候选视点: 16,220  覆盖面片: 354,159
  阶段3  精选航点: 1,314  (压缩至 8.1%)
  阶段4  UAV 1: 288 个视点  路径 425.4 m
  阶段4  UAV 2: 362 个视点  路径 487.0 m
  阶段4  UAV 3: 368 个视点  路径 504.2 m
  阶段4  UAV 4: 296 个视点  路径 476.9 m
  阶段5  CSV: 4 个文件  总 setpoint: 75,132 行
────────────────────────────────────────────────────



In [8]:
inspect_csv(1)   # 查看 UAV 1 的轨迹 CSV 统计


  UAV 1 轨迹统计  ──────────────────────────────
  文件     : output/trajectories/uav_1_trajectory.csv
  总行数   : 16,656
  总时长   : 1665.5s  (27.8 min)
  视点数   : 288

  setpoint 类型分布:
    LAND                         18 行
    LAND_END                      1 行
    LAND_START                    1 行
    TAKEOFF                       8 行
    TAKEOFF_END                   1 行
    TAKEOFF_START                 1 行
    TRANSIT                   2,226 行
    WAYPOINT                 13,824 行
    WAYPOINT_END                288 行
    WAYPOINT_START              288 行

  前 5 行:
  timestamp_s, x, y, z, yaw_rad, type
  0.0, 20.0, 20.0, 0.5, 0.0, TAKEOFF_START
  0.1, 20.0, 20.0, 0.5, 0.0, TAKEOFF
  0.2, 20.0, 20.0, 0.5, 0.0, TAKEOFF
  0.3, 20.0, 20.0, 0.5, 0.0, TAKEOFF
  0.4, 20.0, 20.0, 0.5, 0.0, TAKEOFF



In [ ]:
inspect_csv(2, n_rows=10)   # 查看 UAV 2 的 CSV，预览前10行

---
## 三、参数调整

| 指令 | 说明 |
|---|---|
| `show_config()` | 显示全部配置参数（分类展示）|
| `set_config(KEY=val, ...)` | 运行时修改任意参数（立即生效）|
| `reset_config()` | 恢复 config.py 中的默认值 |

> **调参后重跑范围参考：**  
> 修改成像/安全参数 → `reset(2)` 后 `run("2-5")`  
> 修改多机参数 → `reset(4)` 后 `run("4-5")`  
> 修改轨迹参数 → 直接 `stage5()`

In [ ]:
show_config()   # 显示全部配置参数

In [ ]:
set_config(FOV_DEG=80, CAMERA_DISTANCE=4.0)   # 修改成像参数（需重跑阶段2）

In [ ]:
set_config(FLIGHT_SAFE_RADIUS=1.5, HOVER_TIME=8.0)   # 修改安全距离和悬停时长

In [ ]:
set_config(NUM_UAVS=2, TAKEOFF_POINTS=[[20,0,0.5],[-20,0,0.5]])   # 改为双机

In [ ]:
reset_config()   # 恢复所有参数为 config.py 默认值

---
## 四、检查点管理

| 指令 | 说明 |
|---|---|
| `reset()` | 清除全部检查点和内存缓存 |
| `reset(3)` | 仅清除阶段3及之后的检查点（阶段1、2保留）|
| `save_snapshot("name")` | 将当前检查点打包为命名快照 |
| `load_snapshot("name")` | 从命名快照恢复检查点 |
| `list_snapshots()` | 列出所有已保存快照 |

In [ ]:
reset()   # 清除全部检查点（从头重跑时使用）

In [ ]:
reset(3)   # 仅清除阶段3及之后（调整阶段3参数后使用）

In [ ]:
save_snapshot("fov90_radius3")   # 保存当前实验结果为快照

In [ ]:
load_snapshot("fov90_radius3")   # 从快照恢复（可继续运行下游阶段）

In [ ]:
list_snapshots()   # 查看所有已保存快照

---
## 五、典型调试流程示例

以下是几种常用场景，按顺序运行对应格即可。

In [ ]:
# 场景A：首次运行完整流程
run()

In [ ]:
# 场景B：调整覆盖阈值，只重跑阶段3及之后（阶段1、2无需重跑）
reset(3); run("3-5")

In [ ]:
# 场景C：修改相机视场角，重跑阶段2及之后
set_config(FOV_DEG=80); reset(2); run("2-5")

In [ ]:
# 场景D：保存当前结果，切换参数对比实验
save_snapshot("baseline"); set_config(FOV_DEG=120); reset(2); run("2-5")

In [ ]:
# 场景E：恢复基线实验结果，继续调试阶段5
load_snapshot("baseline"); stage5()